# Cluster Then Classify — Phases 3–4
### LLM cluster labeling, then the downstream classifier

Picks up from `phase2_clustered.csv`, `cluster_labels.npy`,
`embeddings_minilm.npy` and `phase2_human_descriptions.json`.

**The labels stay quarantined.** Nothing in this notebook opens
`_true_labels_DO_NOT_OPEN_UNTIL_PHASE_5.csv`. In particular, the prompt below
never tells the model that these are news articles falling into four
categories — naming the taxonomy would hand it the answer and make Phase 5
meaningless. The model discovers the categories from the documents alone.

In [ ]:
import os, re, json, time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity

RANDOM_STATE = 42
pd.set_option('display.max_colwidth', 140)

In [ ]:
work = pd.read_csv('phase2_clustered.csv')
emb = np.load('embeddings_minilm.npy')
labels = np.load('cluster_labels.npy')
human_desc = json.load(open('phase2_human_descriptions.json'))

assert len(work) == len(emb) == len(labels), 'row counts drifted between phases'
K = len(set(labels))
print(f'{len(work)} rows, {K} clusters')
print(work['cluster'].value_counts().sort_index())

---
## Phase 3 — LLM labeling

### 3.1 Pick representatives

Not a random sample. The documents closest to the centroid are the cluster's
core; a random draw pulls in boundary cases that make coherent clusters look
muddled and push the model toward vague labels.

In [ ]:
def representatives(cluster_id, n=15):
    """The n documents nearest the cluster centroid."""
    idx = np.where(labels == cluster_id)[0]
    centroid = emb[idx].mean(axis=0, keepdims=True)
    sims = cosine_similarity(emb[idx], centroid).ravel()
    return work['text'].iloc[idx[np.argsort(-sims)[:n]]].tolist()


reps = {c: representatives(c, n=15) for c in range(K)}

for c, docs in reps.items():
    print(f'--- cluster {c} (n={(labels == c).sum()}) ---')
    for d in docs[:3]:
        print('   ', d[:110])

### 3.2 Build one prompt for all clusters

Labeling clusters one API call at a time produces collisions — two clusters
both come back "Technology News" because neither call knew the other existed.
Showing all clusters together forces the model to make them mutually exclusive,
which is what you need for a classification target.

In [ ]:
def truncate(s, n=280):
    return s if len(s) <= n else s[:n].rsplit(' ', 1)[0] + '...'


def build_prompt(reps, per_cluster=15):
    blocks = []
    for c, docs in reps.items():
        listed = '\n'.join(f'  {i+1}. {truncate(d)}' for i, d in enumerate(docs[:per_cluster]))
        blocks.append(f'### CLUSTER {c}  (contains {(labels == c).sum()} documents total)\n{listed}')
    corpus = '\n\n'.join(blocks)

    return f"""You are analysing the output of an unsupervised clustering run on a
corpus of short text documents. Each cluster below is shown via the documents
closest to its centre.

Your job is to give every cluster a category label that could be used as a
classification target.

Requirements:
- Labels must be MUTUALLY EXCLUSIVE. No two clusters may receive labels that
  overlap in meaning. If two clusters look similar, find what actually separates
  them and let the labels express that difference.
- Labels must be SHORT: one to three words, title case.
- Base the label on what the documents are ABOUT, not on their writing style,
  their length, or the publication they came from.
- Set "coherent" to false if a cluster has no single subject and reads as a
  grab-bag. Do not invent a label to be agreeable.
- "confidence" is your own 0-1 estimate that this label describes the cluster.

Return ONLY a JSON array. No prose, no markdown fences.

[
  {{"cluster": 0, "label": "...", "description": "one sentence", "coherent": true, "confidence": 0.0}},
  ...
]

{corpus}
"""


prompt = build_prompt(reps)
print(prompt[:1500])
print(f'\n... total prompt length: {len(prompt):,} chars (~{len(prompt)//4:,} tokens)')

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

api_key = os.environ.get('GOOGLE_API_KEY')
if not api_key:
    raise RuntimeError('Set GOOGLE_API_KEY in your environment before running this cell.')

llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    google_api_key=api_key,
    temperature=0,          # labeling should be reproducible; this is not a creative task
)

In [ ]:
def response_text(resp):
    """`.content` is usually a str, but newer versions return a list of blocks."""
    c = resp.content
    if isinstance(c, str):
        return c
    return ''.join(b.get('text', '') if isinstance(b, dict) else str(b) for b in c)


def parse_json(text):
    """Models add markdown fences and preamble no matter how firmly you ask."""
    t = re.sub(r'^```(?:json)?|```$', '', text.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(t)
    except json.JSONDecodeError:
        m = re.search(r'\[.*\]', t, re.DOTALL)     # fall back to the outermost array
        if not m:
            raise
        return json.loads(m.group(0))


def call_llm(prompt, retries=3):
    for attempt in range(retries):
        try:
            raw = response_text(llm.invoke(prompt))
            return parse_json(raw), raw
        except Exception as e:
            if attempt == retries - 1:
                raise
            print(f'  attempt {attempt+1} failed ({type(e).__name__}), retrying...')
            time.sleep(2 ** attempt)

In [ ]:
parsed, raw = call_llm(prompt)

# Always keep the raw response. You will want it for the writeup, and you do not
# want to pay for the call twice when the parser turns out to have eaten a field.
open('phase3_raw_response.txt', 'w').write(raw)
json.dump(parsed, open('phase3_cluster_labels.json', 'w'), indent=2)

llm_labels = pd.DataFrame(parsed).sort_values('cluster').reset_index(drop=True)
llm_labels

### 3.3 Audit before propagating

Three checks, none of which need the true labels. Do them now — a bad label
here silently becomes thousands of bad training rows.

In [ ]:
# Check 1 -- did the model flag anything as incoherent?
bad = llm_labels[(~llm_labels['coherent']) | (llm_labels['confidence'] < 0.6)]
if len(bad):
    print('FLAGGED — read these clusters again before continuing:')
    print(bad[['cluster', 'label', 'coherent', 'confidence']])
else:
    print('All clusters marked coherent with confidence >= 0.6')

# Check 2 -- are the labels actually distinct? Duplicates or near-duplicates mean
# the prompt failed to force exclusivity, or k is too high for the real structure.
names = llm_labels['label'].str.lower().str.strip()
print(f'\n{len(names)} labels, {names.nunique()} unique')
if names.nunique() < len(names):
    print('COLLISION — re-run with a stronger exclusivity instruction, or lower k')

In [ ]:
# Check 3 -- does the model agree with YOU? Your Phase 2 descriptions were
# written before the model saw anything, so this is a real independent check.
compare = llm_labels[['cluster', 'label', 'description']].copy()
compare['your_phase2_description'] = compare['cluster'].map(
    {int(k): v for k, v in human_desc.items()}
)
compare

Read that table row by row. Where the model's label and your own description
disagree, one of you is wrong about the cluster, and finding out which is worth
more than any metric in this notebook.

If they disagree everywhere, do not proceed — go back to Phase 2 and re-examine
k. If they agree everywhere, you are clear to propagate.

In [ ]:
# --- propagate: every document inherits its cluster's label -----------------
label_map = dict(zip(llm_labels['cluster'], llm_labels['label']))

work['llm_label'] = work['cluster'].map(label_map)
work.to_csv('phase3_bootstrapped.csv', index=False)

print(work['llm_label'].value_counts())
work[['text', 'cluster', 'llm_label']].head()

### 3.4 Optional — spot-check the propagation

Cluster-level labels assume every member belongs. Boundary documents often do
not. This asks the model to label individual documents against the taxonomy it
just produced, then measures agreement with the propagated label.

Disagreement rate is a **label-noise estimate you can report without ever
touching ground truth.** Roughly 15 API calls.

In [ ]:
SPOT_N = 300     # documents to check
BATCH = 20       # per API call

taxonomy = '\n'.join(f'- {r.label}: {r.description}' for r in llm_labels.itertuples())
sample = work.sample(SPOT_N, random_state=RANDOM_STATE)

def spot_prompt(texts):
    listed = '\n'.join(f'{i+1}. {truncate(t, 240)}' for i, t in enumerate(texts))
    return f"""Assign each document below to exactly one category.

Categories:
{taxonomy}

Return ONLY a JSON array of objects: [{{"n": 1, "label": "..."}}, ...]
Use the category names exactly as written above.

Documents:
{listed}
"""

preds = []
for start in range(0, len(sample), BATCH):
    chunk = sample.iloc[start:start + BATCH]
    out, _ = call_llm(spot_prompt(chunk['text'].tolist()))
    got = {int(o['n']): o['label'] for o in out}
    preds += [got.get(i + 1) for i in range(len(chunk))]
    print(f'  {start + len(chunk)}/{len(sample)}', end='\r')

sample = sample.assign(llm_direct=preds)
agree = (sample['llm_direct'] == sample['llm_label']).mean()
print(f'\n\npropagated label agrees with direct LLM label: {agree:.1%}')
print(f'estimated label noise from propagation: ~{1 - agree:.1%}')

In [ ]:
# Where does propagation disagree most? These are your leaky cluster boundaries.
disagree = pd.crosstab(sample['llm_label'], sample['llm_direct'])
print(disagree)

print('\nExamples of disagreement:')
for r in sample[sample['llm_direct'] != sample['llm_label']].head(5).itertuples():
    print(f'  cluster says "{r.llm_label}" / direct says "{r.llm_direct}"')
    print(f'    {r.text[:130]}\n')

---
## Phase 4 — Train the downstream classifier

Trained on the bootstrapped labels only. No ground truth is used or available.

Note what this score does and does not mean: the held-out split carries
bootstrapped labels too, so a high number says the classifier learned the
*clustering*, not that it learned the *task*. Phase 5 is what separates those.

In [ ]:
x = work['text']
y = work['llm_label']

xtr, xte, ytr, yte, itr, ite = train_test_split(
    x, y, np.arange(len(work)),
    test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f'train {len(xtr)}, test {len(xte)}')

In [ ]:
# Variant A -- TF-IDF. The cheap deployable: no embedding model at inference.
tfidf_clf = Pipeline([
    ('vec', TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True,
                            stop_words='english')),
    ('clf', LogisticRegression(max_iter=2000, class_weight='balanced')),
])
tfidf_clf.fit(xtr, ytr)

pred_a = tfidf_clf.predict(xte)
print(classification_report(yte, pred_a))

In [ ]:
# Variant B -- embeddings. Usually stronger, but needs MiniLM in production too.
emb_clf = LogisticRegression(max_iter=2000, class_weight='balanced')
emb_clf.fit(emb[itr], ytr)

pred_b = emb_clf.predict(emb[ite])
print(classification_report(yte, pred_b))

In [ ]:
import pickle

pickle.dump(tfidf_clf, open('phase4_tfidf_clf.pkl', 'wb'))
pickle.dump(emb_clf, open('phase4_emb_clf.pkl', 'wb'))
np.save('phase4_test_idx.npy', ite)
np.save('phase4_train_idx.npy', itr)

print('Fidelity to the clustering (NOT task accuracy):')
print(f'  tf-idf     {(pred_a == yte).mean():.3f}')
print(f'  embeddings {(pred_b == yte).mean():.3f}')
print('\nBoth numbers become interpretable only after the Phase 5 reveal.')

---
### Before Phase 5

You should now have, without having looked at a single true label:

- `phase3_cluster_labels.json` — the taxonomy the model discovered
- `phase3_bootstrapped.csv` — every document with an inherited label
- an agreement rate from 3.4, which is your honest label-noise estimate
- two trained classifiers and a frozen train/test split

Freeze all of it. Phase 5 opens the true labels, runs the Hungarian alignment,
and fills in the ablation table — and every one of those numbers is only
trustworthy because none of the decisions above were made with the answers
visible.